# US-004：预处理 — 滤波、降采样、重参考、坏导插值

**目标：** 掌握 EEG 预处理的标准操作，理解每步的物理意义和参数选择。

## 0. 准备数据

In [ ]:
import mne
import numpy as np

# 加载 sample 数据，只取 EEG 通道
sample_dir = mne.datasets.sample.data_path()
raw_fname = sample_dir / "MEG" / "sample" / "sample_audvis_raw.fif"
raw = mne.io.read_raw_fif(raw_fname, preload=True)
raw.pick_types(eeg=True)  # 只保留 EEG 通道
print(f"原始: {raw}")

## 1. 滤波

### 1.1 高通滤波 — 去除慢漂移
截止频率通常选 0.1~1 Hz，去除皮肤电位漂移等低频噪声。

In [ ]:
raw_highpass = raw.copy().filter(l_freq=1.0, h_freq=None)
print(f"高通滤波后 info: highpass={raw_highpass.info['highpass']} Hz")

### 1.2 低通滤波 — 去除高频噪声
截止频率通常 30~40 Hz（保留到 gamma 可设 80-100 Hz）。

In [ ]:
raw_lowpass = raw.copy().filter(l_freq=None, h_freq=40.0)
print(f"低通滤波后 info: lowpass={raw_lowpass.info['lowpass']} Hz")

### 1.3 带通滤波 — 最常用
一次同时做高通和低通。

In [ ]:
raw_bandpass = raw.copy().filter(l_freq=1.0, h_freq=40.0)
print(f"带通 1-40 Hz: {raw_bandpass.info['highpass']}-{raw_bandpass.info['lowpass']} Hz")

### 1.4 陷波滤波 — 去除工频干扰
中国/欧洲 50 Hz，北美 60 Hz。

In [ ]:
# 注意：sample 数据采样率 600Hz，Nyquist=300Hz，50Hz 在范围内
raw_notch = raw.copy().notch_filter(freqs=50.0)
print("50 Hz 陷波完成")

### 1.5 滤波参数详解

| 参数 | 含义 | 典型值 |
|------|------|--------|
| `l_freq` / `h_freq` | 截止频率 | 1 Hz / 40 Hz |
| `l_trans_bandwidth` | 低端过渡带宽 | 'auto' 或 0.5 Hz |
| `h_trans_bandwidth` | 高端过渡带宽 | 'auto' 或 10 Hz |
| `fir_design` | FIR 滤波器设计方法 | 'firwin'（默认） |
| `filter_length` | 滤波器长度 | 'auto'（自动计算） |
| `phase` | 相位 | 'zero'（零相位，默认）|

**关键：** MNE 默认使用零相位 FIR 滤波，不会引入相位延迟。

In [ ]:
# 对比滤波前后的频谱
raw.compute_psd(fmax=80).plot()          # 滤波前
raw_bandpass.compute_psd(fmax=80).plot()  # 滤波后

## 2. 降采样

降低采样率以减少计算量和存储。注意：降采样前必须先低通滤波防止混叠！

In [ ]:
print(f"原始采样率: {raw_bandpass.info['sfreq']} Hz")

# 降采样到 150 Hz（注意先做了 40 Hz 低通，满足 Nyquist）
raw_resampled = raw_bandpass.copy().resample(sfreq=150)
print(f"降采样后采样率: {raw_resampled.info['sfreq']} Hz")
print(f"数据点: {raw.n_times} → {raw_resampled.n_times}")

## 3. 重参考

EEG 测量的是电位差，必须选择参考。常见方式：
- **平均参考**：所有通道平均作为参考
- **乳突参考**：M1/M2 或耳后电极
- **单通道参考**：Cz、FCz 等

In [ ]:
# 方式一：平均参考
raw_avg_ref = raw_resampled.copy().set_eeg_reference(ref_channels='average')
print("已设为平均参考")

# 方式二：指定通道为参考（如双侧乳突）
# raw_mastoid_ref = raw_resampled.copy().set_eeg_reference(ref_channels=['M1', 'M2'])

# 方式三：REST 参考（无穷远参考，需要安装 mne_connectivity 或特定流程）
# REST 是更现代的方法，但需要额外的 BEM 模型

## 4. 坏导检测与插值

坏导 = 信号质量差的通道。MNE 可以手动标记或自动检测，然后用邻居插值。

In [ ]:
# 4.1 手动标记坏导
# 交互模式下可以在 raw.plot() 中点选
# 这里手动演示
raw_avg_ref.info['bads'] = []  # 清空已有坏导
print(f"当前坏导: {raw_avg_ref.info['bads']}")

# 假设某些通道坏了（随机选一个演示）
import random
fake_bad = random.choice(raw_avg_ref.ch_names)
raw_avg_ref.info['bads'] = [fake_bad]
print(f"假设坏导: {raw_avg_ref.info['bads']}")

In [ ]:
# 4.2 自动检测坏导
# MNE 提供多种自动检测方法
# 注意：实际用前先清掉我们模拟的坏导
raw_avg_ref.info['bads'] = []

# 基于 RANSAC 的自动检测
# bads_detected = mne.preprocessing.find_bad_channels_maxwell(
#     raw_avg_ref, return_scores=True, verbose=True
# )
print("自动检测需要实际数据，演示跳过。")

In [ ]:
# 4.3 插值坏导
# 基于球面样条插值，用周围好通道的信号重建坏导
raw_avg_ref.info['bads'] = [fake_bad]  # 恢复我们的假坏导
raw_interp = raw_avg_ref.copy().interpolate_bads(reset_bads=True)
print(f"插值后坏导列表: {raw_interp.info['bads']}")  # 应为空

## 5. 完整预处理 Pipeline

将以上步骤串起来：

In [ ]:
# ===== 一键预处理 =====
def preprocess_pipeline(raw):
    """标准 EEG 预处理流程"""
    # 1. 陷波去工频
    raw = raw.copy().notch_filter(freqs=50)
    # 2. 带通滤波 1-40 Hz
    raw = raw.filter(l_freq=1.0, h_freq=40.0)
    # 3. 降采样
    raw = raw.resample(sfreq=150)
    # 4. 重参考
    raw = raw.set_eeg_reference(ref_channels='average')
    # 5. 插值坏导
    raw.interpolate_bads(reset_bads=True)
    return raw

# 重新加载原始数据测试
raw_test = mne.io.read_raw_fif(raw_fname, preload=True).pick_types(eeg=True)
raw_clean = preprocess_pipeline(raw_test)
print(f"预处理完成: {raw_clean}")
print(f"滤波: {raw_clean.info['highpass']}-{raw_clean.info['lowpass']} Hz")
print(f"采样率: {raw_clean.info['sfreq']} Hz")

## 6. 练习

1. 尝试不同的高通截止频率（0.1, 0.5, 2 Hz），观察 ERP 波形差异
2. 对比平均参考 vs REST 参考的效果
3. 手动添加一段大振幅方波（模拟坏段），看看滤波对它的影响